# Thai Election OCR Pipeline (PaddleOCR)
## Super AI Engineer Season 6 - OCR Competition 2569

Extract structured voting data from scanned Thai election result documents.

**Pipeline:**
1. Download data from Kaggle
2. Group multi-page documents
3. OCR with PaddleOCR (Thai) + coordinate-based table extraction
4. Fuzzy match party names to template
5. Generate submission CSV

**Runtime: GPU (T4)** - ไปที่ Runtime > Change runtime type > GPU

In [ ]:
# === Cell 1: Install Dependencies ===
!pip install -q easyocr rapidfuzz tqdm opencv-python-headless 2>/dev/null
print('Install complete.')

In [ ]:
# === Cell 2: Kaggle Download ===
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)

from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(f'Kaggle credentials set: {username}')

COMPETITION = 'super-ai-engineer-season-6-ocr-2569'
!kaggle competitions download -c {COMPETITION} -p /content/ 2>&1

# Unzip
import glob as glob_module
for zf in glob_module.glob('/content/*.zip'):
    print(f'Extracting: {zf}')
    !unzip -qo "{zf}" -d /content/
print('Done.')

In [ ]:
# === Cell 3: Imports & Configuration ===
import os, re, json, time, cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from tqdm.notebook import tqdm
from rapidfuzz import fuzz, process

# Auto-discover data paths
found = glob_module.glob('/content/**/images', recursive=True)
if found:
    IMAGE_DIR = Path(found[0])
    DATA_DIR = IMAGE_DIR.parent
else:
    DATA_DIR = Path('/content/data')
    IMAGE_DIR = DATA_DIR / 'images'

SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'

# Find submission template
found_tmpl = glob_module.glob('/content/**/submission_template.csv', recursive=True)
SUBMISSION_TEMPLATE = Path(found_tmpl[0]) if found_tmpl else DATA_DIR / 'submission_template.csv'

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'
PROGRESS_FILE = OUTPUT_DIR / 'progress.txt'

# Verify
images = sorted(IMAGE_DIR.glob('*.png'))
print(f'Images: {len(images)}')
print(f'Template: {SUBMISSION_TEMPLATE} (exists={SUBMISSION_TEMPLATE.exists()})')
print(f'Sample labels: {SAMPLE_LABELS_DIR.exists()}')

In [ ]:
# === Cell 4: Load Template & Group Documents ===
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template: {template_df.shape}')
print(template_df.head())

# Group images by doc_id
def group_documents(image_dir):
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')
    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if m:
            doc_id = m.group(1)
            page = int(m.group(2)) if m.group(2) else 1
            docs[doc_id].append((page, str(p)))
    for d in docs:
        docs[d].sort()
    return dict(docs)

documents = group_documents(IMAGE_DIR)
print(f'Documents: {len(documents)}')

# Template lookup
template_doc_rows = template_df.groupby('doc_id').size().to_dict()
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'row_num': row['row_num'],
        'party_name': row['party_name'],
    })
for d in doc_expected:
    doc_expected[d].sort(key=lambda x: x['row_num'])

# Doc types
doc_types = {}
for d in documents:
    if 'party_list' in d.lower() or 'partylist' in d.lower():
        doc_types[d] = 'partylist'
    else:
        doc_types[d] = 'constituency'

c_count = sum(1 for v in doc_types.values() if v == 'constituency')
p_count = sum(1 for v in doc_types.values() if v == 'partylist')
print(f'Constituency: {c_count}, Party list: {p_count}')

In [ ]:
# === Cell 5: Initialize EasyOCR ===
import easyocr

# Thai + English OCR engine (GPU)
ocr_engine = easyocr.Reader(['th', 'en'], gpu=True)
print('EasyOCR initialized (Thai + English, GPU).')

In [ ]:
# === Cell 6: OCR Helper Functions (v2) ===

THAI_TO_ARABIC = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
SKIP_NUMBERS = {'2569', '2570', '2568', '25', '69'}  # years, common garbage

def clean_number(text):
    """Extract clean Arabic number from OCR text."""
    s = str(text).translate(THAI_TO_ARABIC)
    s = re.sub(r'[,.\s]', '', s)  # remove commas, dots, spaces
    s = re.sub(r'[^0-9]', '', s)
    return s if s else ''

def is_valid_vote(num_str):
    """Check if a number string looks like a valid vote count."""
    if not num_str or num_str in SKIP_NUMBERS:
        return False
    return True

def has_thai(text):
    return bool(re.search(r'[\u0e00-\u0e7f]', str(text)))

def ocr_page(img_path, engine):
    """OCR a single page, return text boxes with coordinates."""
    result = engine.readtext(str(img_path), paragraph=False)
    if not result:
        return [], 0, 0
    
    # Get image dimensions
    img = cv2.imread(str(img_path))
    img_h, img_w = img.shape[:2] if img is not None else (3000, 2100)
    
    boxes = []
    for (bbox, text, conf) in result:
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        boxes.append({
            'text': text.strip(),
            'conf': conf,
            'cx': sum(xs) / len(xs),
            'cy': sum(ys) / len(ys),
            'x_min': min(xs), 'x_max': max(xs),
            'y_min': min(ys), 'y_max': max(ys),
            'width': max(xs) - min(xs),
            'height': max(ys) - min(ys),
        })
    return boxes, img_h, img_w

def group_into_rows(boxes, y_threshold=30):
    """Group text boxes into rows by y-coordinate."""
    if not boxes:
        return []
    boxes_sorted = sorted(boxes, key=lambda b: b['cy'])
    rows = []
    current_row = [boxes_sorted[0]]
    for box in boxes_sorted[1:]:
        if abs(box['cy'] - current_row[-1]['cy']) < y_threshold:
            current_row.append(box)
        else:
            rows.append(sorted(current_row, key=lambda b: b['cx']))
            current_row = [box]
    rows.append(sorted(current_row, key=lambda b: b['cx']))
    return rows

print('Helper functions v2 ready.')

In [ ]:
# === Cell 7: Extraction Logic v2 — Party Search Approach ===
# Instead of detecting table rows, SEARCH for known party names in OCR output
# and find the nearest vote number for each.

from rapidfuzz import fuzz, process

def find_vote_near_party(party_box, all_boxes, img_w):
    """Find the best vote count near a matched party box.
    
    Look for numeric text to the RIGHT of the party name, in the same row.
    """
    party_cy = party_box['cy']
    party_cx = party_box['cx']
    
    candidates = []
    for box in all_boxes:
        num = clean_number(box['text'])
        if not num or not is_valid_vote(num):
            continue
        
        # Must be roughly same y (same row) — within 35px
        if abs(box['cy'] - party_cy) > 35:
            continue
        
        # Prefer boxes to the RIGHT of party name
        # But for party list, votes might be further right
        x_dist = box['cx'] - party_cx
        
        # If vote is to the left of party name, skip (likely row number)
        if x_dist < -50:
            continue
            
        # Prefer rightmost number (vote column is typically rightmost)
        candidates.append((box, x_dist, num))
    
    if not candidates:
        return None
    
    # Pick the rightmost valid candidate (most likely to be vote column)
    candidates.sort(key=lambda c: c[0]['cx'], reverse=True)
    return candidates[0][2]


def extract_document_ocr(doc_id, page_paths, doc_type, engine, expected_parties=None):
    """Extract votes using party-name search approach.
    
    For each expected party, find it in the OCR text and grab the nearest vote number.
    """
    if not expected_parties:
        return []
    
    # OCR all pages, collect all boxes
    all_boxes = []
    for page_num, img_path in page_paths:
        boxes, img_h, img_w = ocr_page(img_path, engine)
        
        # Filter: skip top 12% (header) and bottom 12% (signature) of page
        table_boxes = [b for b in boxes if img_h * 0.12 < b['cy'] < img_h * 0.88]
        
        # Add page offset for multi-page docs
        page_offset = (page_num - 1) * img_h
        for b in table_boxes:
            b['cy_global'] = b['cy'] + page_offset
            b['page'] = page_num
        
        all_boxes.extend(table_boxes)
    
    if not all_boxes:
        return []
    
    # Collect all Thai text boxes as party name candidates
    thai_boxes = [b for b in all_boxes if has_thai(b['text']) and len(b['text']) >= 2]
    
    # For each expected party, find best matching Thai text box
    results = {}
    matched_boxes = set()  # prevent double-matching
    
    for party_name in expected_parties:
        best_box = None
        best_score = 0
        
        for i, box in enumerate(thai_boxes):
            if id(box) in matched_boxes:
                continue
            
            # Try multiple fuzzy strategies
            score1 = fuzz.ratio(party_name, box['text'])
            score2 = fuzz.partial_ratio(party_name, box['text'])
            score3 = fuzz.token_sort_ratio(party_name, box['text'])
            score = max(score1, score2, score3)
            
            if score > best_score and score >= 55:
                best_score = score
                best_box = box
        
        if best_box:
            matched_boxes.add(id(best_box))
            
            # Find vote count near this party box (same page only)
            same_page_boxes = [b for b in all_boxes if b.get('page') == best_box.get('page')]
            vote = find_vote_near_party(best_box, same_page_boxes, 2100)
            
            results[party_name] = {
                'party_name': party_name,
                'votes': vote if vote else '0',
                'ocr_text': best_box['text'],
                'match_score': best_score,
            }
    
    # Return in order of expected_parties
    return [results.get(p, {'party_name': p, 'votes': '0', 'ocr_text': '', 'match_score': 0}) 
            for p in expected_parties]

print('Extraction logic v2 ready (party-search approach).')

In [ ]:
# === Cell 8: Test on sample document ===
test_doc = list(documents.keys())[0]
test_type = doc_types[test_doc]
expected_parties = [e['party_name'] for e in doc_expected[test_doc]]

print(f'Testing: {test_doc} ({test_type})')
print(f'Pages: {len(documents[test_doc])}, Expected parties: {len(expected_parties)}')

test_results = extract_document_ocr(
    test_doc, documents[test_doc], test_type, ocr_engine,
    expected_parties=expected_parties
)

print(f'\nExtracted {len(test_results)} rows:')
for r in test_results:
    score = r.get('match_score', 0)
    ocr_txt = r.get('ocr_text', '')[:20]
    marker = '✓' if score >= 70 else '?' if score >= 55 else '✗'
    print(f"  {marker} {r['party_name']:<25s} → votes={r['votes']:>8s}  (score={score:3.0f}, ocr='{ocr_txt}')")

# Compare with sample label
label_file = SAMPLE_LABELS_DIR / f'{test_doc}.json'
if label_file.exists():
    with open(label_file, encoding='utf-8') as f:
        label = json.load(f)
    print(f'\n--- Ground Truth Comparison ---')
    label_map = {}
    for r in label.get('results', []):
        p = r.get('party', r.get('party_name', ''))
        label_map[p] = str(r['votes'])
    
    total_dist = 0
    for r in test_results:
        party = r['party_name']
        pred = r['votes']
        # Find matching label
        actual = '0'
        for lp, lv in label_map.items():
            if fuzz.ratio(party, lp) > 70:
                actual = lv
                break
        from difflib import SequenceMatcher
        dist = sum(1 for a, b in zip(actual.ljust(max(len(actual),len(pred))), 
                                       pred.ljust(max(len(actual),len(pred)))) if a != b)
        # Simple levenshtein
        dist = abs(len(actual) - len(pred)) + sum(1 for a, b in zip(actual, pred) if a != b)
        total_dist += dist
        if actual != pred:
            print(f"  {party:<25s}  actual={actual:>8s}  pred={pred:>8s}  {'✓' if actual==pred else '✗'}")
    print(f'  Score: {total_dist} total edit distance')

In [ ]:
# === Cell 9: Checkpoint & Save Functions (v2) ===

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


def save_submission_csv(results, template_df, doc_expected, output_dir):
    """Generate submission CSV from results (v2 format)."""
    submission_votes = {}
    
    for doc_id, expected_rows in doc_expected.items():
        ocr_rows = results.get(doc_id, [])
        
        if not ocr_rows:
            for exp in expected_rows:
                submission_votes[exp['id']] = '0'
            continue
        
        # New format: results are already ordered by expected_parties
        for i, exp in enumerate(expected_rows):
            if i < len(ocr_rows):
                v = ocr_rows[i].get('votes', '0')
                submission_votes[exp['id']] = v if v else '0'
            else:
                submission_votes[exp['id']] = '0'
    
    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(submission_votes).fillna('0')
    csv_path = output_dir / 'submission.csv'
    sub.to_csv(csv_path, index=False)
    return csv_path


def update_progress(done, total, errors, start_time):
    elapsed = time.time() - start_time
    speed = done / elapsed if elapsed > 0 else 0
    eta = (total - done) / speed if speed > 0 else 0
    
    filled = sum(1 for _ in [])  # placeholder
    with open(PROGRESS_FILE, 'w') as f:
        f.write(f'OCR Progress\n')
        f.write(f'============\n')
        f.write(f'Completed: {done}/{total} documents ({done/total*100:.1f}%)\n')
        f.write(f'Errors: {len(errors)}\n')
        f.write(f'Speed: {speed:.2f} docs/sec\n')
        f.write(f'Elapsed: {elapsed/60:.1f} min\n')
        f.write(f'ETA: {eta/60:.1f} min\n')
        f.write(f'Last updated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')

print('Save functions v2 ready.')

In [ ]:
# === Cell 10: Main Processing Loop (v2) ===
# DELETE old checkpoint to re-run with new logic:
if CHECKPOINT_FILE.exists():
    os.remove(CHECKPOINT_FILE)
    print('Deleted old checkpoint (new extraction logic)')

results = load_checkpoint()
print(f'Loaded {len(results)} existing results from checkpoint.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in results]
print(f'Documents to process: {len(todo)} (skipped {len(results)} already done)')

SAVE_EVERY = 10
errors = []
start_time = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR')):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]
    expected_parties = [e['party_name'] for e in doc_expected[doc_id]]
    
    try:
        rows = extract_document_ocr(
            doc_id, page_paths, doc_type, ocr_engine,
            expected_parties=expected_parties
        )
        results[doc_id] = rows
        
        # Count how many got votes (not '0')
        matched = sum(1 for r in rows if r.get('votes', '0') != '0')
        if matched < len(expected_parties) * 0.5:
            print(f'  WARN {doc_id}: only {matched}/{len(expected_parties)} parties matched')
    
    except Exception as e:
        errors.append(doc_id)
        print(f'  ERROR {doc_id}: {e}')
        import traceback; traceback.print_exc()
        results[doc_id] = []
    
    # Periodic save
    done = len(results)
    if done % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
        update_progress(done, len(documents), errors, start_time)
        elapsed = time.time() - start_time
        print(f'  [{done}/{len(documents)}] {elapsed/60:.1f}min elapsed, CSV saved')

# Final save
save_checkpoint(results)
save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
update_progress(len(results), len(documents), errors, start_time)

elapsed = time.time() - start_time
print(f'\nDone! {len(results)}/{len(documents)} docs in {elapsed/60:.1f} min')
print(f'Errors: {len(errors)}')
if errors:
    print(f'Failed: {errors}')
print(f'Submission: {OUTPUT_DIR}/submission.csv')

In [ ]:
# === Cell 11: Validate Against Sample Labels ===

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[j]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

results = load_checkpoint()
votes_map = match_and_build_votes(results, template_df, doc_expected)

if SAMPLE_LABELS_DIR.exists():
    total_dist = 0
    total_pairs = 0
    
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        doc_id = jf.stem
        label_results = label.get('results', [])
        
        print(f'\n=== {doc_id} ===')
        
        # Match label parties to template
        expected_rows = doc_expected.get(doc_id, [])
        
        for label_row in label_results:
            actual = str(label_row['votes'])
            party = label_row.get('party', label_row.get('party_name', ''))
            
            # Find this party's submission id
            predicted = '0'
            for exp in expected_rows:
                if fuzz.ratio(party, exp['party_name']) > 70:
                    predicted = votes_map.get(exp['id'], '0')
                    break
            
            dist = levenshtein(actual, predicted)
            total_dist += dist
            total_pairs += 1
            
            if dist > 0:
                print(f'  {party:30s}  actual={actual:>8s}  pred={predicted:>8s}  dist={dist}')
    
    if total_pairs > 0:
        print(f'\n=== Sample Validation ===')
        print(f'Pairs: {total_pairs}')
        print(f'Mean Levenshtein: {total_dist/total_pairs:.4f}')
else:
    print('No sample labels found.')

In [ ]:
# === Cell 12: Review Results ===
results = load_checkpoint()

# Stats
row_counts = {d: len(r) for d, r in results.items()}
expected_counts = {d: template_doc_rows.get(d, 0) for d in results}

match_count = sum(1 for d in results if row_counts[d] == expected_counts.get(d, -1))
mismatch = {d: (row_counts[d], expected_counts[d]) for d in results if row_counts[d] != expected_counts.get(d, -1)}

print(f'Total docs: {len(results)}')
print(f'Row count match: {match_count}/{len(results)}')
print(f'Row count mismatch: {len(mismatch)}')

if mismatch:
    print('\nMismatches (doc: got/expected):')
    for d, (got, exp) in sorted(mismatch.items())[:20]:
        print(f'  {d}: {got}/{exp}')

In [ ]:
# === Cell 13: Download Submission ===
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))

In [ ]:
# === Cell 14: Debug — See raw OCR output for a document ===
DEBUG_DOC = list(documents.keys())[0]  # Change this
page_paths = documents[DEBUG_DOC]

print(f'Document: {DEBUG_DOC}')
print(f'Pages: {len(page_paths)}\n')

for page_num, img_path in page_paths:
    print(f'=== Page {page_num}: {img_path} ===')
    boxes, img_h, img_w = ocr_page(img_path, ocr_engine)
    print(f'Image: {img_w}x{img_h}, Boxes: {len(boxes)}')
    
    # Show all boxes sorted by y then x
    boxes_sorted = sorted(boxes, key=lambda b: (b['cy'], b['cx']))
    for b in boxes_sorted:
        num = clean_number(b['text'])
        is_num = f' [NUM={num}]' if num else ''
        is_th = ' [TH]' if has_thai(b['text']) else ''
        zone = 'HEADER' if b['cy'] < img_h * 0.12 else 'FOOTER' if b['cy'] > img_h * 0.88 else 'TABLE'
        print(f"  y={b['cy']:6.0f} x={b['cx']:6.0f} [{zone:6s}] '{b['text']}'{is_num}{is_th}")
    print()